<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/GPT_4_1_mini_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning GPT-4.1-mini in Our Project (Supervised Fine-Tuning)

We first run **GPT-4.1 and GPT-4.1-mini** through the course RAG pipeline as pre-fine-tuning baselines, then **supervised fine-tune** `gpt-4.1-mini-2025-04-14` on labelled question-and-answer pairs generated from the AI-tutor knowledge base, and finally bring the fine-tuned model back — **closed book, no retrieved context** — to compare against those baselines. In between we run the whole managed fine-tuning lifecycle in code: **build the dataset → validate → upload → train → monitor → use**.

**Vendor note:** deliberately single-vendor — managed fine-tuning of an OpenAI model is an OpenAI-API exercise, so the three-provider setup cell does not apply here (models and prices as of **August 2026**).

🖥️ *Runs in **Google Colab or locally** — the only key needed is `OPENAI_API_KEY` (Colab Secrets 🔑, an environment variable, or a `.env` file).*

> ⚠️ **Fine-tuning availability (checked August 21, 2026).** OpenAI announced on **May 7, 2026** that it is winding down its self-serve fine-tuning platform: organizations that had never run a fine-tune lost the ability to create jobs that day, organizations with no fine-tuned-model inference in the previous 60 days lost it on **July 2, 2026**, and the remaining active organizations lose job creation on **January 6, 2027**. Already fine-tuned models keep serving until their base model is deprecated. The SFT API below is unchanged and `gpt-4.1-mini-2025-04-14` is still a listed SFT model for organizations that retain access — the Towards AI org does, so this notebook runs end to end for the course. **If your org cannot create fine-tuning jobs, you can still run the baselines and build and validate the dataset (Sections 3–5)** — read Sections 6–7, the training job and the fine-tuned inference, as a walkthrough: every idea there (dataset quality, the supervised method, the validation split, loss curves) transfers directly to the local LoRA/QLoRA fine-tuning lesson at the end of this section, which anyone can run on a free Colab GPU.

## 🧭 What You'll Learn

- What **supervised fine-tuning (SFT)** actually is — labelled prompt-and-response pairs the model learns to imitate — and why it fits this project: a consistent tutor voice and course-specific recent material
- Recording **pre-fine-tuning baselines** — the untuned GPT-4.1 and GPT-4.1-mini in the course RAG pipeline, and the mini once more *without* context — so "did fine-tuning help?" has a reference point
- Turning a raw document corpus into an SFT **chat-format JSONL dataset** programmatically: chunk → sample (a fixed quota from each of the four most recent sources) → a teacher model writes grounded Q&A pairs
- Why **the quality of the pairs is the whole game**: the model becomes whatever its labels are
- Validating a dataset *before* you pay to train on it: schema checks, token counts, API minimums, and a held-out **validation split**
- Creating a fine-tuning job with the **`method` parameter set explicitly to `supervised`**, polling it without hiding errors, and reading the loss curves
- What fine-tuning **does not** fix — and why production still pairs the tuned model with retrieval

## 1. Setup: Environment, Keys, and Knobs

The standard course setup cell, minus the provider dropdown (this lesson is OpenAI-only by nature). One key, pinned installs in Colab, `.env` locally.

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API key
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

REQUIRED_KEYS = ["OPENAI_API_KEY"]

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (July 2026).
    # huggingface_hub (for the prebuilt vector store download) is preinstalled in Colab.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "openai==2.46.0",
            "chromadb==1.5.9",
            "tiktoken==0.13.0",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

    # In Colab: Secrets tab (🔑 icon) → Add new secret → OPENAI_API_KEY
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    # Locally: dependencies are installed once from the repo's requirements.
    # Keys live in a .env file at the repo root.
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

from openai import OpenAI

client = OpenAI()

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'}")

✅ Setup complete — local


In [2]:
# @title ⚙️ Models and dataset knobs { display-mode: "form" }

# The model we FINE-TUNE. Snapshot pinned on purpose: fine-tuning targets an exact
# dated snapshot, and the resulting model id embeds it (docs: supervised fine-tuning guide).
FT_MODEL = "gpt-4.1-mini-2025-04-14"  # @param ["gpt-4.1-mini-2025-04-14", "gpt-4.1-2025-04-14"] {allow-input: true}

# The TEACHER that writes the training answers (course-standard OpenAI model).
TEACHER_MODEL = "gpt-5.6-luna"  # @param ["gpt-5.6-luna", "gpt-4.1"] {allow-input: true}

# The two models the inference comparison keeps from the original lesson.
GPT_LARGE = "gpt-4.1"
GPT_SMALL = "gpt-4.1-mini"

# Dataset size — a SAMPLE of the corpus, not the whole thing, so a full run stays
# in the ~1-dollar range. Training pairs are drawn as a FIXED QUOTA PER SOURCE
# (the dataset's own `source` field), from the four most recent corpus sources —
# the material gpt-4.1-mini's April 2025 training most plausibly predates.
# 4 × 25 = 100 training pairs: the original lesson's size, right in the
# "improvements from fine-tuning on 50–100 examples" band OpenAI's SFT guide
# describes (minimum: 10).
TRAIN_SOURCES = ["claude_code_docs", "deep_agents", "agentic_ai_engineering", "langgraph"]
N_PER_SOURCE = 25  # @param {type:"integer"}
N_TRAIN_EXAMPLES = N_PER_SOURCE * len(TRAIN_SOURCES)
N_VAL_EXAMPLES = 30  # @param {type:"integer"}
VAL_DOC_FRACTION = 0.2  # held-out share of documents (all sources) reserved for validation

N_EPOCHS = 2  # passes over the training set (kept from the original lesson)

# ⚠️ The generated pairs are cached to disk (Section 4.3). If you change the knobs
# above, delete sft_train_pairs.json / sft_val_pairs.json to regenerate.
print(f"fine-tune {FT_MODEL} | teacher {TEACHER_MODEL} | "
      f"{len(TRAIN_SOURCES)} sources × {N_PER_SOURCE} = {N_TRAIN_EXAMPLES} train / {N_VAL_EXAMPLES} val | {N_EPOCHS} epochs")

fine-tune gpt-4.1-mini-2025-04-14 | teacher gpt-5.6-luna | 4 sources × 25 = 100 train / 30 val | 2 epochs


## 2. Supervised Fine-Tuning: What It Is, and Why Here

**Supervised fine-tuning (SFT)** is the simplest fine-tuning method OpenAI's platform offers, and the one this whole notebook uses: you hand the API a file of **labelled examples** — complete chat conversations where the assistant message is the answer you *want* — and gradient descent nudges the model's weights until its own answers imitate those labels. That's it. No reward model, no preference pairs, no grader: the label **is** the supervision, which is exactly what "supervised" means. (The API also offers DPO and reinforcement fine-tuning as separate `method` types; we deliberately don't use them — for teaching a voice and a body of factual material from worked examples, imitation learning is the right tool, and in Section 6 you'll see us set `method.type = "supervised"` explicitly.)

**Why fine-tune here at all?** Two reasons, both specific to this project:

1. **A consistent tutor voice.** The course's AI tutor should answer in the same persona, scope, and style every time. Prompting gets you most of the way; SFT bakes the behaviour into the weights, so the model answers as the tutor even with a minimal prompt.
2. **Recent, course-specific material.** Our `ai_tutor_knowledge` corpus holds current documentation — LangChain, LangGraph, Claude Code, Deep Agents, the OpenAI docs — much of it newer than what `gpt-4.1-mini` (an April 2025 snapshot) saw in training. Training on Q&A pairs derived from that corpus moves some of it into the model.

**And what fine-tuning does *not* fix.** SFT on 100 examples will not turn a mini model into an encyclopedia, and a fine-tuned model still has a knowledge cutoff — the corpus updates weekly; the model's weights don't. It can still hallucinate, it cannot cite sources, and it doesn't know which document an answer came from. That is why Section 7 demos the fine-tuned model **closed book** — the cleanest way to see what actually moved into the weights — while production keeps a tuned model **paired with retrieval**: fine-tuning shapes *how* the model answers and seeds what it knows; retrieval supplies *what is true right now*, scoped and citable. Treat them as complements, never substitutes.

📎 *The distillation frame from the original lesson survives one model swap: a stronger, cheaper-per-answer teacher (`gpt-5.6-luna`, the course's OpenAI default) writes the reference answers, and the small student (`gpt-4.1-mini`) learns to imitate them — knowledge distilled from teacher to student through plain SFT. (The teacher is no longer the same model family as the student; what matters for SFT is only the quality of the labels it writes.)*

## 3. Baselines Before Fine-Tuning: GPT-4.1 and GPT-4.1-mini

Before training anything, record what the **untuned** models do — otherwise *"did fine-tuning help?"* has no answer. This is the original lesson's comparison, kept intact (with the model pair moved from its GPT-4o generation to **GPT-4.1 and GPT-4.1-mini**): the course's prebuilt vector store, the plain-Python `ask()` — embed the question, retrieve the top-5 chunks from Chroma, answer from the context — and one fixed demo question, answered by the big model, then the small one, then the small one again **without any retrieved context**.

The demo question asks about the **Model Context Protocol (MCP)** on purpose: it is recent material, and Section 4 draws the training sample from the corpus's four most recent sources — the Claude Code documentation among them, which is where the MCP setup material lives — so Section 7 can ask this exact question again, closed book, and read the difference as the fine-tune's doing.

The pairing is also the lesson's economics in miniature: GPT-4.1 at $2.00 / $8.00 per 1M tokens versus GPT-4.1-mini at $0.40 / $1.60 — 5× cheaper (August 2026). The bet this notebook tests is that a *fine-tuned* mini closes some of the quality gap while keeping most of that price advantage.

### 3.1 Load the Prebuilt Vector Store

The course's prebuilt store — the AI-tutor knowledge base, already chunked and embedded with `text-embedding-3-small` — from the Towards AI org dataset on Hugging Face: download once, unzip, open. The paths are relative, so this works the same in Colab and locally, and every cell is safe to re-run.

In [3]:
# Download the prebuilt vector store from the Hugging Face hub (course org dataset)
from huggingface_hub import hf_hub_download

vectorstore = hf_hub_download(
    repo_id="towardsai-tutors/full-stack-ai-engineering-data",
    filename="vector_stores/ai_tutor_knowledge-text-embedding-3-small-1536d-0e801a3f.zip",
    repo_type="dataset",
    local_dir=".",
)
print(vectorstore)

vector_stores/ai_tutor_knowledge-text-em(…): reconstructing file:   0%|          |  0.00B / 97.4MB            

vector_stores/ai_tutor_knowledge-text-em(…): downloading bytes:           |  0.00B            

/Users/jai/Documents/code-repo/ai-tutor-rag-system/notebooks/vector_stores/ai_tutor_knowledge-text-embedding-3-small-1536d-0e801a3f.zip


In [4]:
import pathlib
import zipfile

STORE_DIR = pathlib.Path("ai_tutor_knowledge-text-embedding-3-small-1536d")

if not STORE_DIR.exists():  # extract once; safe to re-run
    with zipfile.ZipFile(vectorstore) as zf:
        zf.extractall(".")
print("Vector store ready at", STORE_DIR.resolve())

Vector store ready at /Users/jai/Documents/code-repo/ai-tutor-rag-system/notebooks/ai_tutor_knowledge-text-embedding-3-small-1536d


In [5]:
# Setup an Embedding Model (plain OpenAI SDK — must match the store: text-embedding-3-small, 1536d)


def embed_query(text):
    return client.embeddings.create(model="text-embedding-3-small", input=text).data[0].embedding

In [6]:
import chromadb

# Open the prebuilt Chroma collection (the AI-tutor knowledge base, embedded once)
db = chromadb.PersistentClient(path=str(STORE_DIR / "chroma"))
chroma_collection = db.get_collection("ai_tutor_knowledge")
print(chroma_collection.count(), "chunks")

7428 chunks


In [7]:
def ask(question, model, top_k=5, temperature=1):
    result = chroma_collection.query(query_embeddings=[embed_query(question)], n_results=top_k)
    sources = [
        {"id": id_, "text": text, "score": 1 - dist, "metadata": meta}
        for id_, text, dist, meta in zip(
            result["ids"][0], result["documents"][0], result["distances"][0], result["metadatas"][0]
        )
    ]
    context = "\n\n---\n\n".join(src["text"] for src in sources)
    completion = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": "Answer the question using only the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return completion.choices[0].message.content, sources

### 3.2 Response Generation Using GPT-4.1

In [8]:
# Query (RAG, BASELINE — before any fine-tuning): top-5 chunks, answered by GPT-4.1
response_gpt_4_1, sources_gpt_4_1 = ask(
    "What is the Model Context Protocol, and how do I connect Claude Code to an MCP server?",
    model=GPT_LARGE,
)

response_gpt_4_1

'**Model Context Protocol (MCP)** is an open-source standard protocol that allows large language models (LLMs) like Claude Code to interact with external tools and data sources through structured API calls. Think of MCP as a “USB-C port for AI”: it provides a standardized way for tools, databases, platforms, and APIs to connect to AI models, so they can access live information, automate workflows, or manipulate data without requiring manual copy-paste or context injection.\n\n**Key features of MCP:**\n- Act as a standardization layer for AI-tool integrations.\n- Support three capability types: invoking tools (functions), accessing resources (like files or databases), and using reusable prompt templates.\n- Enable automation for tasks such as implementing features from issue trackers, analyzing monitoring data, querying databases, integrating design changes, automating emails, and reacting to external events.\n\n---\n\n## How to Connect Claude Code to an MCP server\n\nYou can connect Cl

In [9]:
for src in sources_gpt_4_1:
    print("Chunk ID\t", src["id"])
    print("Title\t", src["metadata"]["title"])
    print("Text\t", src["text"])
    print("Score\t", round(src["score"], 4))
    print("Metadata\t", src["metadata"])
    print("-_" * 20)

Chunk ID	 51feaab1-a4fc-502f-bb46-72a26a8f3a9e-0
Title	 Model Context Protocol
Text	 # Model Context Protocol

Model Context Protocol (MCP) connects models to tools and context. Use it to give Codex access to third-party documentation, or to let it interact with developer tools like your browser or Figma.

Codex supports MCP servers in both the CLI and the IDE extension.

## Supported MCP features

- **STDIO servers**: Servers that run as a local process (started by a command).
  - Environment variables
- **Streamable HTTP servers**: Servers that you access at an address.
  - Bearer token authentication
  - OAuth authentication (run `codex mcp login <server-name>` for servers that support OAuth)

## Connect Codex to an MCP server

Codex stores MCP configuration in `config.toml` alongside other Codex configuration settings. By default this is `~/.codex/config.toml`, but you can also scope MCP servers to a project with `.codex/config.toml` (trusted projects only).

The CLI and the IDE ex

### 3.3 Response Generation Using GPT-4.1-mini

In [10]:
# Query (RAG, BASELINE — before any fine-tuning): same retrieval, answered by GPT-4.1-mini
response_gpt_4_1_mini, sources_gpt_4_1_mini = ask(
    "What is the Model Context Protocol, and how do I connect Claude Code to an MCP server?",
    model=GPT_SMALL,
)

response_gpt_4_1_mini

'The Model Context Protocol (MCP) is an open-source, standard protocol that allows Large Language Models (LLMs) like Claude Code to connect and interact with external tools, data sources, and services through structured API calls. MCP acts as a standardized interface (like a "USB-C port" for AI) enabling AI applications to access resources such as issue trackers, databases, monitoring dashboards, and more, seamlessly integrating these external contexts into AI workflows.\n\nMCP operates via a client-server architecture with three components:\n\n- **MCP Hosts**: Applications (like Claude desktop or IDEs) that want to access external data or tools.\n- **MCP Clients**: Protocol clients managing connections between hosts and servers.\n- **MCP Servers**: Lightweight services exposing tools, resources, or prompts over the MCP standard.\n\n---\n\n**How to connect Claude Code to an MCP server:**\n\n1. **Find an MCP server** to connect with or build your own following the [MCP server guide](htt

In [11]:
for src in sources_gpt_4_1_mini:
    print("Chunk ID\t", src["id"])
    print("Title\t", src["metadata"]["title"])
    print("Text\t", src["text"])
    print("Score\t", round(src["score"], 4))
    print("Metadata\t", src["metadata"])
    print("-_" * 20)

Chunk ID	 51feaab1-a4fc-502f-bb46-72a26a8f3a9e-0
Title	 Model Context Protocol
Text	 # Model Context Protocol

Model Context Protocol (MCP) connects models to tools and context. Use it to give Codex access to third-party documentation, or to let it interact with developer tools like your browser or Figma.

Codex supports MCP servers in both the CLI and the IDE extension.

## Supported MCP features

- **STDIO servers**: Servers that run as a local process (started by a command).
  - Environment variables
- **Streamable HTTP servers**: Servers that you access at an address.
  - Bearer token authentication
  - OAuth authentication (run `codex mcp login <server-name>` for servers that support OAuth)

## Connect Codex to an MCP server

Codex stores MCP configuration in `config.toml` alongside other Codex configuration settings. By default this is `~/.codex/config.toml`, but you can also scope MCP servers to a project with `.codex/config.toml` (trusted projects only).

The CLI and the IDE ex

### 3.4 Response Generation for the Same Query, Without Context

Same question, **no retrieval**: this is the "before" picture the fine-tune has to beat. Watch what the untuned mini does on its own — generic definitions, hedges, or confidently invented specifics (**your output may differ**). Section 7 asks the fine-tuned model this same question under the same no-context conditions.

In [12]:
def ask_without_context(question, model, temperature=1):

    completion = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": "Answer the question, Don't hallucinate."},
            {"role": "user", "content": f"Question: {question}"},
        ],
    )
    return completion.choices[0].message.content



response_gpt_4_1 = ask_without_context(
    "What is the Model Context Protocol, and how do I connect Claude Code to an MCP server?",
    model=GPT_LARGE,
)

response_gpt_4_1

'**Model Context Protocol (MCP)** is an emerging protocol designed to standardize how language models (such as Anthropic’s Claude, OpenAI’s GPT, etc.) interact with external tools, knowledge bases, and other software. MCP provides a common interface for exchanging context, messages, and commands between a model and external systems. The goal is to improve interoperability, traceability, and fine-grained tool use with language models.\n\n### What is the Model Context Protocol (MCP)?\n- **Purpose:** Provides a standardized way for ML models to interact with external tools, context servers, or plugins.\n- **How it works:** Models send/receive context (such as user queries, code, document text, etc.) and associated actions (like function calls) via structured messages (e.g., JSON over HTTP/websockets).\n- **Use Cases:** Tool use, agentic workflows, plug-and-play integrations, enhanced retrieval augmented generation (RAG), etc.\n\n### "Claude Code" and MCP\n- **Claude Code** refers to code 

In [13]:
response_gpt_4_1_mini_no_context = ask_without_context(
    "What is the Model Context Protocol, and how do I connect Claude Code to an MCP server?",
    model=GPT_SMALL,
)

response_gpt_4_1_mini_no_context

'The **Model Context Protocol (MCP)** is a communication protocol designed to facilitate interactions between AI models (like Claude from Anthropic) and external servers or applications. It enables models to connect with external environments, exchange information, and extend their capabilities beyond isolated operation.\n\n### What is the Model Context Protocol (MCP)?\n\n- MCP is a protocol that standardizes how AI models interact with external services.\n- It provides a way to manage context, send and receive data, and synchronize state between the model and an external server.\n- By using MCP, you can connect AI models like Claude Code to a server that supports MCP, allowing for richer integrations, real-time data exchange, and customized workflows.\n\n### Connecting Claude Code to an MCP Server\n\nTo connect Claude Code to an MCP server, you typically need to follow these steps:\n\n1. **Set Up an MCP Server**\n   - Ensure you have an MCP server running. This server will handle comm

**What just happened?** Three baselines on the record, all answering the same MCP question: the flagship **with** retrieval (`response_gpt_4_1`), the mini **with** retrieval (`response_gpt_4_1_mini`), and the mini **without** it (`response_gpt_4_1_mini_no_context`). With the documents in the prompt, both models answer from the retrieved MCP pages — retrieval was never the suspense. Strip the context away and the mini is on its own; read that last answer closely for hedges and invented specifics. It is the exact "before" picture Section 7's fine-tuned model has to beat.

## 4. Build the SFT Training Dataset

The supervised fine-tuning API takes JSONL in the **chat-completions format** — one complete conversation per line (docs: *supervised fine-tuning guide*):

```json
{"messages": [{"role": "system", "content": "..."},
              {"role": "user", "content": "..."},
              {"role": "assistant", "content": "..."}]}
```

The model learns to produce the assistant message given everything before it. So building a dataset means answering three questions, and each gets a cell below:

1. **Where do the documents come from?** The updated `ai_tutor_knowledge` corpus — 788 documents across seven sources, the same file behind the course's retrieval lessons and prebuilt vector stores.
2. **How does a document become a training example?** We chunk each document with the course chunker, sample chunks as a fixed quota per source from the four most recent sources, and have the **teacher model write one Q&A pair per chunk** — the question a student might ask, and an answer written *only from that chunk*. Grounding the teacher in the excerpt is what keeps the labels factual instead of invented: the teacher isn't asked what it knows, it's asked to teach what the document says.
3. **What goes in the system message?** The course tutor persona — pulled from the notebook that defined it, not invented here (see Section 4.3).

**The quality of these pairs is what the model learns — nothing else.** SFT has no notion of "truth", only of imitation: feed it vague answers and it learns to be vague; feed it answers grounded in current documentation and it learns those facts and that style. In real projects this step — not the training call — is where fine-tunes are won or lost, which is why we spot-check pairs against their source chunks before paying to train.

### 4.1 Load the Corpus

In [14]:
import json
import pathlib
import random

import requests

# 📎 The course download cell, unchanged — dataset hosted in the Towards AI org
#    dataset repo on Hugging Face (788 documents: langchain, llama_index, openai_docs,
#    claude_code_docs, langgraph, deep_agents, agentic_ai_engineering).
DATA_URL = "https://huggingface.co/datasets/towardsai-tutors/full-stack-ai-engineering-data/resolve/main/ai_tutor_knowledge.jsonl"
DATA_PATH = pathlib.Path("ai_tutor_knowledge.jsonl")

if not DATA_PATH.exists():
    DATA_PATH.write_bytes(requests.get(DATA_URL, timeout=180).content)
    print("Downloaded", DATA_PATH)

with open(DATA_PATH, encoding="utf-8") as f:
    docs_all = [json.loads(line) for line in f]

from collections import Counter

print(f"{len(docs_all)} documents")
for source, count in Counter(d["source"] for d in docs_all).most_common():
    print(f"  {source:24} {count:>4}")

788 documents
  langchain                 221
  llama_index               206
  openai_docs               155
  claude_code_docs          100
  langgraph                  39
  deep_agents                37
  agentic_ai_engineering     30


### 4.2 Chunk, Split by Document, Sample

Documents are too long to ground one Q&A pair each, so we cut them with the **course chunker** (512-token chunks, 128 overlap — the same contract as every retrieval lesson) and sample chunks across the whole corpus, so all seven sources contribute.

Two rules, both borrowed from the embedding fine-tuning lesson, both about honesty:

- **Split by document, *before* generating anything.** Validation pairs come from documents the training pairs never touch. Adjacent chunks share 128 overlapping tokens, so a chunk-level split would leak nearly identical text across the boundary — and the validation loss would flatter you.
- **Seeded sampling.** The split and the sample are reproducible, so a rerun builds the same dataset instead of a subtly different one.

And one addition for this lesson: the training sample is **quota'd by the dataset's own `source` field** rather than drawn uniformly. The corpus is lopsided — langchain (221 docs), llama_index (206), and openai_docs (155) dwarf langgraph (39), deep_agents (37), and agentic_ai_engineering (30) — so a uniform 100-chunk draw from ~7,400 chunks would be dominated by the big framework docs and could miss the small recent sources entirely. Instead, the cell below takes **`N_PER_SOURCE` (25) pairs from each of the four most recent sources** — `claude_code_docs`, `deep_agents`, `agentic_ai_engineering`, `langgraph` — the material an April 2025 snapshot most plausibly predates, and the home of the Section 3 demo question (Claude Code's MCP setup docs). This is the general recipe for steering an SFT dataset: **decide the source mix deliberately — never leave it to sampling luck.** (Which individual pages land in each 25 is still up to the seed — read the split printout, and spot-check in 4.3.)

In [15]:
import tiktoken

# 📎 chunk() from "From Script to Pipeline" — the course chunker, cl100k_base by
#    course convention (matches every retrieval lesson and the prebuilt stores).
#    Billing/token *counts* for the fine-tune use the GPT-4.1 family's own
#    o200k_base encoding instead — that happens in Section 5.
ENC = tiktoken.get_encoding("cl100k_base")


def chunk(text, chunk_size=512, chunk_overlap=128):
    """Split text into token-based chunks; consecutive chunks share chunk_overlap tokens."""
    tokens = ENC.encode(text)
    chunks = []
    step = chunk_size - chunk_overlap
    for start in range(0, len(tokens), step):
        window = tokens[start : start + chunk_size]
        chunks.append(ENC.decode(window))
        if start + chunk_size >= len(tokens):
            break  # the final window reached the end of the text
    return chunks


def to_records(documents):
    """Chunk every document into records with stable ids and source metadata."""
    records = []
    for doc in documents:
        for i, piece in enumerate(chunk(doc["content"])):
            records.append({
                "id": f"{doc['doc_id']}-{i}",
                "text": piece,
                "title": doc["name"],
                "url": doc["url"],
                "source": doc["source"],
            })
    return records


# Split DOCUMENTS first (seeded), so validation pairs come from documents the
# training set never saw — overlapping chunks make a chunk-level split leaky.
# The validation slice is held out from the WHOLE corpus (all seven sources);
# training then draws only on the four TRAIN_SOURCES, a fixed quota from each.
docs = docs_all.copy()
random.Random(42).shuffle(docs)
split_at = int(len(docs) * (1 - VAL_DOC_FRACTION))
train_pool_docs, val_docs = docs[:split_at], docs[split_at:]

# Chunk each training source separately (the dataset's own `source` field is the
# selector — no other filtering), and plan this source's share of the quota.
candidate_groups = []
train_chunks = []
print(f"{'source':26} {'docs':>5} {'chunks':>7} {'pairs planned':>14}")
for seed, source in enumerate(TRAIN_SOURCES, start=10):
    source_docs = [d for d in train_pool_docs if d["source"] == source]
    assert source_docs, f"No documents with source == {source!r} — check TRAIN_SOURCES for typos."
    source_chunks = to_records(source_docs)
    train_chunks.extend(source_chunks)

    # We oversample by 1.5× so Section 4.3 can skip the occasional failed
    # generation and still hit the target counts.
    n_target = min(N_PER_SOURCE, len(source_chunks))
    candidates = random.Random(seed).sample(source_chunks, min(int(n_target * 1.5), len(source_chunks)))
    candidate_groups.append((candidates, n_target))
    print(f"{source:26} {len(source_docs):>5} {len(source_chunks):>7} {n_target:>14}")

val_chunks = to_records(val_docs)
val_candidates = random.Random(1).sample(val_chunks, min(int(N_VAL_EXAMPLES * 1.5), len(val_chunks)))

print(f"\ntrain: {sum(t for _, t in candidate_groups)} pairs planned from {len(TRAIN_SOURCES)} sources")
print(f"val:   {len(val_docs):>3} held-out documents (all sources) → {len(val_chunks):>5} chunks → {len(val_candidates)} sampled candidates")

source                      docs  chunks  pairs planned
claude_code_docs              84    1458             25
deep_agents                   30     388             25
agentic_ai_engineering        26     658             25
langgraph                     27     356             25

train: 100 pairs planned from 4 sources
val:   158 held-out documents (all sources) →  1336 chunks → 45 sampled candidates


### 4.3 The Teacher Writes Grounded Q&A Pairs

Now the pair generation itself. Three deliberate choices:

- **The system message is the course tutor persona — not a new one.** It is copied verbatim from `ask_ai_tutor()` in `Intro_to_Large_Language_Models.ipynb` (Section 1, *"How To Use LLMs via API"*), the cell that defined how our tutor behaves: answer AI questions in an informative, accurate way; refuse everything else. The user-message template comes from the same function. A fine-tune is conditioned on the exact prompt format it saw in training, so whatever we train with here is what we must send at inference in Section 7 — reusing the course's established persona makes that alignment free.
- **The teacher sees *only* the chunk.** The generation prompt hands the teacher one excerpt and instructs it to write a question that excerpt can answer and an answer that uses *only* facts stated in it. That is the grounding mechanism: answers imitate the corpus, not the teacher's memory.
- **Structured output, strictly parsed.** The teacher replies through a Pydantic schema (`question`, `answer`) via `client.chat.completions.parse`, so a malformed generation is an exception we count and skip — never a silently broken line in the training file.

💰 *Cost: one teacher call per pair — ~130 calls at the default knobs (up to ~195 if some generations are skipped), a few minutes and roughly $0.10–0.30 at `gpt-5.6-luna` prices ($0.20 / $1.20 per 1M tokens, August 2026). Both pair sets are cached to disk, so reruns are free.*

In [16]:
from pydantic import BaseModel
from tqdm.auto import tqdm

# 📎 Persona and answering convention from ask_ai_tutor() in
#    Intro_to_Large_Language_Models.ipynb (Section 1, "How To Use LLMs via API") —
#    the cell that defined the course tutor. Reused VERBATIM: these exact strings
#    go into every training example, and again at inference time in Section 7.
TUTOR_SYSTEM = (
    "You are an AI tutor specialized in answering artificial intelligence-related questions. "
    "Only answer AI-related questions, else say that you cannot answer this question."
)
TUTOR_PROMPT = (
    "Please provide an informative and accurate answer to the following question.\n"
    "Question: {question}\nAnswer:"
)


class TutorPair(BaseModel):
    """One SFT training example, as the teacher writes it."""

    question: str
    answer: str


PAIR_GEN_SYSTEM = (
    "You write training data for an AI tutor. Every answer must be grounded strictly "
    "in the excerpt you are given — never in your own prior knowledge."
)

PAIR_GEN_PROMPT = """Documentation excerpt (from "{title}", source: {source}):
---------------------
{text}
---------------------
Write ONE training example for an AI-tutor assistant, using ONLY the excerpt above.

1. question — a self-contained question a student could ask about this material. It must be
   understandable without seeing the excerpt: name the library, tool, or concept explicitly,
   and never refer to "the excerpt", "the context", or "the text".
2. answer — an informative, accurate, tutor-style answer (3–6 sentences), using ONLY facts
   stated in the excerpt. If the excerpt contains code, you may include a short snippet from it.

Do not invent facts the excerpt does not state. If a detail is not in the excerpt, leave it out."""


def make_sft_pairs(candidates, target_count):
    """Teacher-generate up to target_count grounded (question, answer) pairs.

    Iterates an oversampled candidate list so occasional failed generations
    (refusals, schema mismatches) can be skipped — counted, never silent —
    while still reaching the target."""
    pairs, skipped = [], 0
    progress = tqdm(candidates, desc="generating pairs")
    for rec in progress:
        if len(pairs) >= target_count:
            break
        try:
            completion = client.chat.completions.parse(
                model=TEACHER_MODEL,
                messages=[
                    {"role": "system", "content": PAIR_GEN_SYSTEM},
                    {"role": "user", "content": PAIR_GEN_PROMPT.format(**rec)},
                ],
                response_format=TutorPair,
            )
            pair = completion.choices[0].message.parsed
        except Exception as exc:  # a malformed/refused generation: skip it, say so
            skipped += 1
            print(f"  skipped one candidate ({type(exc).__name__}: {exc})")
            continue
        pairs.append({
            "question": pair.question,
            "answer": pair.answer,
            "chunk_id": rec["id"],  # provenance: which chunk grounds this pair
            "source": rec["source"],
        })
        progress.set_postfix(kept=len(pairs), skipped=skipped)
    return pairs


def build_pairs(path, candidate_groups):
    """Generate once, cache to JSON, reload on every later run (and re-pay nothing).

    candidate_groups: list of (candidates, target_count) — generated in order,
    one quota group per source (built in Section 4.2)."""
    path = pathlib.Path(path)
    target_total = sum(target for _, target in candidate_groups)
    if path.exists():
        pairs = json.loads(path.read_text(encoding="utf-8"))
        print(f"Loaded {len(pairs)} pairs from {path}")
    else:
        pairs = []
        for i, (candidates, target) in enumerate(candidate_groups):
            # This group's share, plus any shortfall left by earlier groups —
            # so one thin source degrades gracefully instead of failing the run.
            later = sum(t for _, t in candidate_groups[i + 1 :])
            pairs.extend(make_sft_pairs(candidates, target_total - len(pairs) - later))
        # Cache FIRST — these pairs are paid for, never throw them away.
        path.write_text(json.dumps(pairs, indent=2), encoding="utf-8")
        print(f"Generated and saved {len(pairs)} pairs to {path}")
    if len(pairs) < target_total:
        print(
            f"⚠️ Only {len(pairs)}/{target_total} pairs available — a source's chunk pool "
            f"may be smaller than its quota, or generations were skipped. Proceeding with "
            f"what exists (counts are shown again in Section 4.4); delete {path} to "
            f"regenerate, or lower N_PER_SOURCE."
        )
    return pairs


# Train: one quota group per source (Section 4.2). Val: one general group.
train_pairs = build_pairs("sft_train_pairs.json", candidate_groups)[:N_TRAIN_EXAMPLES]
val_pairs = build_pairs("sft_val_pairs.json", [(val_candidates, N_VAL_EXAMPLES)])[:N_VAL_EXAMPLES]

generating pairs:   0%|          | 0/37 [00:00<?, ?it/s]

generating pairs:   0%|          | 0/37 [00:00<?, ?it/s]

generating pairs:   0%|          | 0/37 [00:00<?, ?it/s]

generating pairs:   0%|          | 0/37 [00:00<?, ?it/s]

Generated and saved 100 pairs to sft_train_pairs.json


generating pairs:   0%|          | 0/45 [00:00<?, ?it/s]

Generated and saved 30 pairs to sft_val_pairs.json


In [17]:
# Spot-check grounding: read a generated pair NEXT TO the chunk it came from.
# This eyeball test is the cheapest quality gate in the whole notebook — if the
# answer states things the chunk does not, fix the generation prompt, not the model.
by_id = {r["id"]: r for r in train_chunks}
sample_pair = train_pairs[0]
grounding_chunk = by_id[sample_pair["chunk_id"]]

print("SOURCE CHUNK —", grounding_chunk["title"], f"({grounding_chunk['source']})")
print(grounding_chunk["text"][:600], "…\n")
print("─" * 70)
print("Q:", sample_pair["question"])
print("A:", sample_pair["answer"])

SOURCE CHUNK — Discover and install prebuilt plugins through marketplaces (claude_code_docs)
> ## Documentation Index
> Fetch the complete documentation index at: https://code.claude.com/docs/llms.txt
> Use this file to discover all available pages before exploring further.

# Discover and install prebuilt plugins through marketplaces

> Find and install plugins from marketplaces to extend Claude Code with new skills, agents, and capabilities.

Plugins extend Claude Code with skills, agents, hooks, and MCP servers. Plugin marketplaces are catalogs that help you discover and install these extensions without building them yourself.

Looking to create and distribute your own marketplace? …

──────────────────────────────────────────────────────────────────────
Q: How do I install a plugin from the official Anthropic marketplace in Claude Code, and what should I do if Claude Code cannot find it?
A: The official Anthropic marketplace, `claude-plugins-official`, is automatically available wh

**What just happened?** The teacher turned sampled corpus chunks into labelled Q&A pairs — the *supervision* in supervised fine-tuning — and cached both sets to disk. Read a few more pairs against their chunks before moving on: every fact the fine-tune will "know" is being decided right here, by these labels, and five minutes of reading now is cheaper than a mistrained model later.

### 4.4 Format as Chat JSONL and Show the Split

Each pair becomes one line in the API's chat format: the **course tutor system message**, the **Section 1 question template** as the user turn, and the teacher's grounded answer as the assistant turn — the message the model learns to produce. We write `sft_train.jsonl` and `sft_val.jsonl` with plain `json.dumps`, one object per line.

In [18]:
def to_chat_example(pair):
    """One SFT example in the chat-completions format the fine-tuning API expects."""
    return {
        "messages": [
            {"role": "system", "content": TUTOR_SYSTEM},
            {"role": "user", "content": TUTOR_PROMPT.format(question=pair["question"])},
            {"role": "assistant", "content": pair["answer"]},
        ]
    }


train_examples = [to_chat_example(p) for p in train_pairs]
val_examples = [to_chat_example(p) for p in val_pairs]

TRAIN_FILE = pathlib.Path("sft_train.jsonl")
VAL_FILE = pathlib.Path("sft_val.jsonl")

for path, examples in [(TRAIN_FILE, train_examples), (VAL_FILE, val_examples)]:
    with open(path, "w", encoding="utf-8") as f:
        for example in examples:
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

# Show the split — sizes and where each side draws from.
print(f"train: {len(train_examples):>3} examples → {TRAIN_FILE}   (quota'd from {len(TRAIN_SOURCES)} sources)")
print(f"val:   {len(val_examples):>3} examples → {VAL_FILE}   (from {len(val_docs)} held-out documents)")
print("\ntrain examples per source:", dict(Counter(p["source"] for p in train_pairs)))
print("val   examples per source:", dict(Counter(p["source"] for p in val_pairs)))
print("\nOne formatted training example:")
print(json.dumps(train_examples[0], indent=2, ensure_ascii=False)[:900], "…")

train: 100 examples → sft_train.jsonl   (quota'd from 4 sources)
val:    30 examples → sft_val.jsonl   (from 158 held-out documents)

train examples per source: {'claude_code_docs': 25, 'deep_agents': 25, 'agentic_ai_engineering': 25, 'langgraph': 25}
val   examples per source: {'langgraph': 8, 'deep_agents': 2, 'agentic_ai_engineering': 6, 'claude_code_docs': 5, 'langchain': 2, 'openai_docs': 4, 'llama_index': 3}

One formatted training example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are an AI tutor specialized in answering artificial intelligence-related questions. Only answer AI-related questions, else say that you cannot answer this question."
    },
    {
      "role": "user",
      "content": "Please provide an informative and accurate answer to the following question.\nQuestion: How do I install a plugin from the official Anthropic marketplace in Claude Code, and what should I do if Claude Code cannot find it?\nAnswer:"
    },
    {
      "role": "

**What just happened?** 130 grounded pairs became two JSONL files in the exact schema the API trains on: 100 training examples, and 30 validation examples from documents the training set never touched. The training side draws a fixed quota from each of the four most recent sources — read the per-source counts above against the plan; the validation side is a held-out slice of the whole corpus, so validation loss stays a read on *general* tutor ability. And the validation file is not decoration — Section 6 hands it to the job, and OpenAI computes a **validation loss** on it during training, which is how you see the model *generalizing* to the tutor task rather than memorizing 100 strings.

## 5. Validate and Upload the Dataset

Never upload a training file you haven't validated — the job's `validating_files` stage will reject a malformed file anyway, but only after you've waited in the queue, and it says less about *why* than your own checks will. Three checks before any upload:

1. **Schema** — every line is a dict with a well-formed `messages` list, valid roles, string contents, and at least one assistant message. The checks below are OpenAI's own, from the cookbook's *[Data preparation for chat model fine-tuning](https://developers.openai.com/cookbook/examples/chat_finetuning_data_prep)*.
2. **Token counts** — counted with the GPT-4.1 family's own `o200k_base` encoding. Billing is per trained token (`total tokens × epochs`), and per-example limits exist (the guide last documented 65,536 tokens per example for this family; oversized examples are truncated during training — ours are chunk-grounded and hundreds of times smaller).
3. **API minimums** — at least **10 examples** per file (docs: *supervised fine-tuning guide*, which also notes improvements typically show from 50–100 examples).

In [19]:
# Format error checks — OpenAI's own validation logic, from the cookbook:
# https://developers.openai.com/cookbook/examples/chat_finetuning_data_prep
from collections import defaultdict


def validate_dataset(examples, label):
    format_errors = defaultdict(int)

    for ex in examples:
        if not isinstance(ex, dict):
            format_errors["data_type"] += 1
            continue

        messages = ex.get("messages", None)
        if not messages:
            format_errors["missing_messages_list"] += 1
            continue

        for message in messages:
            if "role" not in message or "content" not in message:
                format_errors["message_missing_key"] += 1

            if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
                format_errors["message_unrecognized_key"] += 1

            if message.get("role", None) not in ("system", "user", "assistant", "function"):
                format_errors["unrecognized_role"] += 1

            content = message.get("content", None)
            function_call = message.get("function_call", None)

            if (not content and not function_call) or not isinstance(content, str):
                format_errors["missing_content"] += 1

        if not any(message.get("role", None) == "assistant" for message in messages):
            format_errors["example_missing_assistant_message"] += 1

    if format_errors:
        print(f"❌ {label}: found errors:")
        for k, v in format_errors.items():
            print(f"   {k}: {v}")
    else:
        print(f"✅ {label}: no format errors")
    return dict(format_errors)


train_errors = validate_dataset(train_examples, "train")
val_errors = validate_dataset(val_examples, "val")
assert not train_errors and not val_errors, "Fix the format errors above before uploading."

✅ train: no format errors
✅ val: no format errors


In [20]:
# Token counts — with the GPT-4.1 family's own encoding (o200k_base), not the
# cl100k_base we chunked with: chunking follows the course contract, but BILLING
# follows the model being trained. Per-message overheads follow the cookbook's
# num_tokens_from_messages() and are close approximations for budgeting.
ENC_FT = tiktoken.encoding_for_model(FT_MODEL)  # → o200k_base for gpt-4.1-mini
print("counting with encoding:", ENC_FT.name)


def n_tokens_in_example(example, tokens_per_message=3, tokens_per_name=1):
    total = 3  # every reply is primed with <|start|>assistant<|message|>
    for message in example["messages"]:
        total += tokens_per_message
        for key, value in message.items():
            total += len(ENC_FT.encode(str(value)))
            if key == "name":
                total += tokens_per_name
    return total


def dataset_token_report(examples, label):
    counts = sorted(n_tokens_in_example(ex) for ex in examples)
    total = sum(counts)
    print(f"{label}: {len(examples)} examples | tokens — total {total:,}, "
          f"min {counts[0]}, median {counts[len(counts) // 2]}, max {counts[-1]}")
    return total


train_tokens = dataset_token_report(train_examples, "train")
val_tokens = dataset_token_report(val_examples, "val")

# The API minimum (docs: supervised fine-tuning guide): at least 10 examples.
MIN_EXAMPLES = 10
assert len(train_examples) >= MIN_EXAMPLES, f"Need ≥{MIN_EXAMPLES} training examples."

# Billing estimate: trained tokens ≈ dataset tokens × epochs. The job reports the
# real number afterwards as job.trained_tokens — we check our estimate against it.
LAST_PUBLISHED_TRAIN_PRICE = 5.00  # $ per 1M trained tokens for gpt-4.1-mini — the LAST
# published rate: OpenAI removed gpt-4.1 fine-tuning prices from its live pricing page
# during the platform wind-down, so treat this as an estimate and verify billed
# amounts in your dashboard (checked August 21, 2026).
est_trained_tokens = train_tokens * N_EPOCHS
print(f"\nestimated trained tokens: {train_tokens:,} × {N_EPOCHS} epochs = {est_trained_tokens:,}")
print(f"estimated training cost:  ~${est_trained_tokens / 1e6 * LAST_PUBLISHED_TRAIN_PRICE:.2f} "
      f"(at the last-published ${LAST_PUBLISHED_TRAIN_PRICE:.2f}/1M rate — verify in your dashboard)")

counting with encoding: o200k_base
train: 100 examples | tokens — total 18,894, min 116, median 190, max 249
val: 30 examples | tokens — total 5,503, min 133, median 188, max 223

estimated trained tokens: 18,894 × 2 epochs = 37,788
estimated training cost:  ~$0.19 (at the last-published $5.00/1M rate — verify in your dashboard)


**What just happened?** Both files passed the schema the API enforces, we know their exact token weight in the *right* tokenizer, and the training run is priced before a single token is billed. This order — validate, count, estimate, *then* upload — is the habit worth keeping; the alternative is discovering a malformed line or a surprise bill from inside a queued job.

Now upload both files with `purpose="fine-tune"`. The API stores them and hands back file IDs the job will reference:

In [21]:
train_file = client.files.create(file=open(TRAIN_FILE, "rb"), purpose="fine-tune")
val_file = client.files.create(file=open(VAL_FILE, "rb"), purpose="fine-tune")

print("training file:  ", train_file.id, f"({train_file.bytes:,} bytes)")
print("validation file:", val_file.id, f"({val_file.bytes:,} bytes)")

training file:   file-TusgXXW8b6eg3YJRqipSi6 (100,442 bytes)
validation file: file-CwnjVoaP31DoPnrSzaBz5e (29,431 bytes)


## 6. Run the Supervised Fine-Tuning Job


**Runtime:** the job passes through `validating_files → queued → running` before a terminal state. A job this size typically completes in roughly **10–40 minutes**, most of it queue time, but queue load varies — once training starts, the job object's `estimated_finish` field gives a live estimate, and the polling cell below prints it.

The creation call itself is three ideas:

- `model` is the **dated snapshot** `gpt-4.1-mini-2025-04-14` — fine-tunes target exact snapshots, and the resulting model id embeds it.
- `method` is set to **`supervised` explicitly**. This is the API's name for SFT — the deliberate choice this notebook is about (the deprecated top-level `hyperparameters` argument predates the `method` block; current docs put hyperparameters *inside* `method.supervised`). `n_epochs=2` is spelled out; `batch_size` and `learning_rate_multiplier` stay `"auto"` by omission.
- `validation_file` makes OpenAI score the held-out set during training, and `seed`/`suffix` make the run reproducible and the model findable.

In [22]:
# Create the supervised fine-tuning job.
# ⚠️ This is the cell that starts billing — everything above it was preparation.
job = client.fine_tuning.jobs.create(
    model=FT_MODEL,                    # the dated snapshot we fine-tune
    training_file=train_file.id,
    validation_file=val_file.id,       # held-out docs → validation loss during training
    suffix="ai-tutor",                 # → ft:gpt-4.1-mini-2025-04-14:<your-org>:ai-tutor:<id>
    seed=42,                           # same seed + same data → reproducible job (docs)
    method={
        "type": "supervised",          # ← SFT, explicitly — the method this lesson teaches
        "supervised": {
            "hyperparameters": {"n_epochs": N_EPOCHS},  # batch_size / LR multiplier: "auto"
        },
    },
)

print("job id: ", job.id)
print("status: ", job.status)
print("method: ", job.method.type)

PermissionDeniedError: Error code: 403 - {'error': {'message': 'OpenAI is winding down the fine-tuning platform and your organization is no longer able to create new fine-tuning training jobs. Learn more https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning', 'type': 'invalid_request_error', 'param': None, 'code': 'training_not_available'}}

### 6.1 Monitor the Job — Explicitly, Without Hiding Errors

Fine-tuning is asynchronous: the call above only *queued* work. The loop below polls every 30 seconds and prints a timestamped status line, so the wait is visible instead of a silent hang. Two deliberate choices, both the opposite of the classic tutorial loop:

- **No blanket `try/except`.** If polling itself fails (bad key, network down), you want the exception, not a printed shrug that kills the loop while the job runs on unseen.
- **Failures are surfaced loudly.** A `failed` job carries a structured `job.error` (code, message, and often the offending parameter) — the loop prints it and raises, rather than letting a dead job read like a finished one.

*(To stop a running job: `client.fine_tuning.jobs.cancel(job.id)`. Closing the notebook does not cancel it — the job runs server-side.)*

In [ ]:
import time
from datetime import datetime

TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}  # full lifecycle:
# validating_files → queued → running → succeeded | failed | cancelled

while True:
    job = client.fine_tuning.jobs.retrieve(job.id)  # poll errors propagate — on purpose

    stamp = datetime.now().strftime("%H:%M:%S")
    eta = (f" | est. finish {datetime.fromtimestamp(job.estimated_finish):%H:%M:%S}"
           if job.estimated_finish else "")
    trained = f" | trained_tokens {job.trained_tokens:,}" if job.trained_tokens else ""
    print(f"[{stamp}] status: {job.status}{eta}{trained}")

    if job.status in TERMINAL_STATUSES:
        break
    time.sleep(30)

if job.status == "failed":
    # Surface the structured error the API attached — never swallow it.
    err = job.error
    raise RuntimeError(
        f"Fine-tuning job {job.id} FAILED — code={err.code!r}, param={err.param!r}: {err.message}"
    )

print(f"\n🏁 Job {job.status}.")
if job.status == "succeeded":
    print("fine-tuned model:", job.fine_tuned_model)

In [ ]:
# The job's event log — the play-by-play the dashboard shows, fetched in code.
# (The API returns newest first; we reverse into chronological order. Training
# progress events include step-level loss in their messages.)
events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=50)

for event in reversed(list(events)):
    print(f"{datetime.fromtimestamp(event.created_at)}  {event.message}")

In [ ]:
# What did training actually cost? The job reports the real billed quantity.
job = client.fine_tuning.jobs.retrieve(job.id)

if job.trained_tokens:
    print(f"trained_tokens (billed): {job.trained_tokens:,}  "
          f"(our Section 5 estimate: {est_trained_tokens:,})")
    print(f"→ ~${job.trained_tokens / 1e6 * LAST_PUBLISHED_TRAIN_PRICE:.2f} "
          f"at the last-published ${LAST_PUBLISHED_TRAIN_PRICE:.2f}/1M rate — "
          "verify the billed amount in your dashboard")

### 6.2 🔬 OPTIONAL EXPERIMENT — Read the Loss Curves

The job's `result_files` hold a CSV of step-level training metrics — training loss, and validation loss for the held-out file we attached. Falling training loss says the model is learning the labels; the validation curve says whether that learning **generalizes** to pairs from documents it never trained on. Watch for the classic divergence: training loss still falling while validation loss turns upward means the model has begun memorizing your 100 examples — the signal to use fewer epochs, not more.

In [ ]:
# 🔬 OPTIONAL EXPERIMENT — plot the loss curves from the job's result file
import io

import pandas as pd

if job.status == "succeeded" and job.result_files:
    raw = client.files.content(job.result_files[0]).read()
    metrics = pd.read_csv(io.BytesIO(raw))
    print("metrics columns:", metrics.columns.tolist())

    import matplotlib.pyplot as plt

    loss_cols = [c for c in metrics.columns if c.endswith("loss")]
    steps = metrics["step"] if "step" in metrics.columns else metrics.index
    for col in loss_cols:
        series = metrics.assign(_step=steps).dropna(subset=[col])
        plt.plot(series["_step"], series[col], label=col)
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(f"Fine-tuning loss — {job.fine_tuned_model}")
    plt.legend()
    plt.show()

    display(metrics.tail(5))
else:
    print("No result files to read (job not succeeded yet).")

**What just happened?** You ran the full managed-training lifecycle in code — upload, an explicitly `supervised` job, visible polling, surfaced errors, real billed tokens, loss curves — the exact loop the OpenAI dashboard wraps in a UI. The job's product is a **model id** like `ft:gpt-4.1-mini-2025-04-14:<your-org>:ai-tutor:<id>`: a private model only your org can call, served by OpenAI like any other model. Section 7 puts it to work.

## 7. Inference on the Fine-Tuned Model

The fine-tuned model id comes straight off the succeeded job. **Every student gets their own id** — the `ft:` name below encodes the base snapshot, *your* org, our `ai-tutor` suffix, and a unique job hash, and it is callable only by the org that trained it (which is why this cell no longer hard-codes the Towards AI id the original lesson shipped).

In [ ]:
# The model id minted by YOUR job in Section 6. If you fine-tuned earlier (or in
# the dashboard UI), paste your own id instead — the placeholder pattern below is
# what it looks like; it will not run as-is.
FINE_TUNED_MODEL = job.fine_tuned_model
# FINE_TUNED_MODEL = "ft:gpt-4.1-mini-2025-04-14:your-org:ai-tutor:XXXXXXXX"  # ← paste yours

assert FINE_TUNED_MODEL, "No fine-tuned model id — did the job succeed?"
print("using:", FINE_TUNED_MODEL)

### 7.1 Closed-Book: Did the Fine-Tune Take?

First, the cleanest possible read on what SFT changed: the **same held-out validation questions**, asked with **no retrieved context at all**, to the base `gpt-4.1-mini` and the fine-tuned model side by side — next to the teacher-written reference answer each label came from.

One detail matters more than it looks: we ask through the **exact system message and question template the model was trained with** (Section 4.3). A fine-tune is conditioned on its training prompt format — evaluate it in a different format and you are partly measuring the mismatch, not the model. Look for the fine-tuned answers to sound like the tutor labels — scoped, structured, grounded in the corpus's terminology — where the base model answers from generic training knowledge. **Your outputs will differ from any captures shown in the lesson** (temperature 1, your own generated dataset), and on some questions the two models will tie: 100 examples shape a voice and seed some facts, they do not rewrite a model.

In [ ]:
def ask_tutor(question, model, temperature=1):
    """One closed-book tutor turn — the SAME system message and question template
    the model was fine-tuned on (📎 persona from Intro_to_Large_Language_Models.ipynb)."""
    completion = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": TUTOR_SYSTEM},
            {"role": "user", "content": TUTOR_PROMPT.format(question=question)},
        ],
    )
    return completion.choices[0].message.content


# Two held-out questions — from documents (and labels) the model never trained on.
for pair in val_pairs[:2]:
    print("=" * 80)
    print("QUESTION (held-out):", pair["question"])
    print("\n--- reference answer (teacher label, grounded in the corpus) ---")
    print(pair["answer"])
    print(f"\n--- {GPT_SMALL} (base, closed book) ---")
    print(ask_tutor(pair["question"], model=GPT_SMALL))
    print(f"\n--- {FINE_TUNED_MODEL} (fine-tuned, closed book) ---")
    print(ask_tutor(pair["question"], model=FINE_TUNED_MODEL))
    print()

**What just happened?** A three-way read: the label the teacher wrote, what the base model says on its own, and what the fine-tuned model says after two epochs of imitating 100 such labels. Where the fine-tuned answer tracks the reference's substance and voice on material the base model waffles on, that is the fine-tune working; where both models answer alike, you have found the honest edge of what 100 examples buy.

### 7.2 The Baseline Question, Closed Book

Now the payoff the baselines set up: the **same MCP question** from Section 3, asked with **no retrieved context** — the exact condition the untuned mini faced in Section 3.4. Nothing to lean on: whatever the model says here comes from its weights, which now include two epochs over training pairs drawn from the corpus's recent sources — the Claude Code documentation, where the MCP setup material lives, among them. We ask through `ask_tutor()` — the training prompt format — for the same same-format reason as 7.1.

One honesty note before you read the answer: this is a *memorization-flavoured* demo by design. The sources this question lives in were deliberately placed in the training sample, so this shows the material **went in** — 7.1's held-out questions are what measure generalization. And it is still not how you would ship the tutor: no citations, and the corpus will keep updating while these weights stand still. Production pairs the tuned model with retrieval; this cell isolates what the fine-tune itself contributed.

In [ ]:
# The Section 3 question again — answered by the FINE-TUNED model with NO
# retrieved context: weights only, in the tutor format it was trained on.
response_fine_tuned_model = ask_tutor(
    "What is the Model Context Protocol, and how do I connect Claude Code to an MCP server?",
    model=FINE_TUNED_MODEL,
)

print(response_fine_tuned_model)

In [ ]:
# The four answers to the one demo question, side by side:
# Section 3's three baselines, then the fine-tune — closed book.
for name, answer in [
    (f"{GPT_LARGE} + retrieval (baseline)", response_gpt_4_1),
    (f"{GPT_SMALL} + retrieval (baseline)", response_gpt_4_1_mini),
    (f"{GPT_SMALL}, no context (baseline)", response_gpt_4_1_mini_no_context),
    (f"{FINE_TUNED_MODEL}, no context", response_fine_tuned_model),
]:
    print("=" * 80)
    print(f"### {name}\n")
    print(answer)

**What just happened?** Four answers to one question. The two retrieval-backed baselines show what *any* capable model does when the documents are in the prompt — that bar was never in question. The pair to read closely is the **last two**: the untuned mini with no context against the fine-tuned mini with no context. Where the untuned answer stayed generic, hedged, or invented specifics, does the fine-tuned one now sound like the corpus's actual MCP documentation? That difference — and only that difference — is what the fine-tune bought (**your outputs will differ**). The production stance stands: weights for voice and baked-in knowledge, retrieval for freshness, scope, and citations.

## 8. Clean Up (Optional)

A fine-tuned model lives on your org until you delete it (deleting requires the org owner role). The training and validation files persist on your account too:

In [ ]:
# Delete a fine-tuned model (must be an owner of the org the model was created in)
# result_delete = client.models.delete(FINE_TUNED_MODEL)
# print(result_delete)

# The uploaded dataset files also persist on your account until deleted:
# client.files.delete(train_file.id)
# client.files.delete(val_file.id)

## 🧪 Your Turn

- **Easy:** re-read three random pairs from `sft_train_pairs.json` next to their source chunks (Section 4.3's spot-check). Would *you* accept these as labels? Delete the two cache files and regenerate with an edited `PAIR_GEN_PROMPT` if not.
- **Medium:** ask both models (base vs fine-tuned, closed book) three questions of your own about MCP, LangGraph, or Claude Code — recent corpus material. Where does the fine-tune's knowledge end?
- **Hard:** re-run the whole notebook with `N_EPOCHS = 1` and compare the two jobs' loss curves (Section 6.2). Which epoch count would you ship, and what in the validation curve says so?

## 🔑 Key Takeaways

- **SFT = imitation of labelled examples.** The API's `method: {"type": "supervised"}` trains the model to reproduce your assistant messages, nothing more — which is why it is the right tool for a consistent tutor voice and course-specific material, and the wrong tool to expect encyclopedic knowledge from.
- **The dataset is the model.** Chunk → sample → teacher writes grounded Q&A → strict validation. Every fact the fine-tune "learns" was decided by those labels; spot-checking pairs against source chunks is the highest-leverage minute in the pipeline — and when specific material matters, **pin the source mix**: here, a fixed 25-pair quota from each of the four most recent sources, rather than hoping uniform sampling finds them.
- **Persona consistency is free if you don't invent one:** train on the course's established system message and question template, then serve with the same ones — a fine-tune is conditioned on its training format.
- **Validate before you upload, split before you generate.** Schema checks and `o200k_base` token counts price the run in advance; a document-level held-out split is what makes the validation loss mean generalization instead of leakage.
- **Poll explicitly; surface errors.** `validating_files → queued → running → succeeded/failed/cancelled`, with `job.error` printed and raised on failure — never a silent `except: break`.
- **Fine-tuning and RAG are complements.** The closed-book demo shows what moved into the weights — but the tuned model still can't cite sources or track a weekly-updated corpus, so in production it belongs back inside the retrieval pipeline, where the real economics question is a tuned mini vs a 5×-pricier base flagship.
- ⏭️ **OpenAI's managed fine-tuning is sunsetting** (no new orgs since May 7, 2026; job creation ends January 6, 2027) — the next lesson fine-tunes an *open* model with LoRA/QLoRA on a free Colab GPU: same SFT idea, weights you keep, no platform to lose.